# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [20]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [21]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/agedpayables"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 5

--- AgedPayables_20260207170508.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260207170624.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325043000.parquet ---
  Registros: 10
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325043100.parquet ---
  Registros: 10
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325214230.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [22]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 5

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260207170624.parquet
  ✅ Mesmas colunas (1 colunas)
  ✅ Tipos de dados idênticos

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260325043000.parquet
  ✅ Mesmas colunas (1 colunas)
  ❌ Tipos de dados diferem:
     Coluna 'Row': array<struct<ColData:array<struct<id:string,value:string>>,Summary:struct<ColData:array<struct<value:string>>>,group:string,type:string>> → array<struct<Header:struct<ColData:array<struct<value:string>>>,Rows:struct<Row:array<struct<ColData:array<struct<value:string>>>>>,Summary:struct<ColData:array<struct<value:string>>>>>

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260325043100.parquet
  ✅ Mesmas colunas (1 colunas)
  ❌ Tipos de dados diferem:
     Coluna 'Row': array<struct<ColData:array<struct<id:string,value:string>>,Summary:struct<ColData:array<struct<value:string>>>,group:string,type:string>> → array<struct<Header:struct<ColDat

In [23]:
import json
import pandas as pd
from IPython.display import display

def extrair_linhas(row_data, grupo_pai=None):
    """Extrai recursivamente as linhas de dados de um relatório QuickBooks."""
    linhas = []
    rows = row_data.get("Row", [])

    for row in rows:
        nome_grupo = grupo_pai
        if "Header" in row and "Rows" in row:
            col_data = row["Header"].get("ColData", [])
            if col_data and col_data[0].get("value"):
                nome_grupo = col_data[0]["value"]
            linhas.extend(extrair_linhas(row["Rows"], nome_grupo))
        elif "ColData" in row:
            valores = [item.get("value", "") for item in row["ColData"]]
            linhas.append({"grupo": nome_grupo, "valores": valores})
        elif "Header" in row and "ColData" in row.get("Header", {}):
            col_data = row["Header"].get("ColData", [])
            valores = [item.get("value", "") for item in col_data]
            linhas.append({"grupo": nome_grupo, "valores": valores})

    return linhas

# Extrair dados de todos os arquivos e juntar em uma tabela sem duplicatas
tabelas = []

for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON, pulando ---")
        continue

    row = info["df"].first()

    cols_json = json.loads(row["Columns"])
    # Renomear colunas sem título usando ColType ou índice
    nomes_colunas = []
    for idx, c in enumerate(cols_json.get("Column", [])):
        titulo = c.get("ColTitle", "").strip()
        if not titulo:
            # Usar ColType como fallback, ou "Coluna_N"
            titulo = c.get("ColType", f"Coluna_{idx}")
        nomes_colunas.append(titulo)

    header = json.loads(row["Header"])
    report_name = header.get("ReportName", "?")
    end_period = header.get("EndPeriod", "?")
    print(f"--- {nome} | {report_name} | Período: {end_period} ---")

    rows_json = json.loads(row["Rows"])
    linhas = extrair_linhas(rows_json)
    print(f"  Linhas extraídas: {len(linhas)}")

    if linhas:
        dados = []
        for linha in linhas:
            registro = {"Grupo": linha["grupo"]}
            for i, col_nome in enumerate(nomes_colunas):
                registro[col_nome] = linha["valores"][i] if i < len(linha["valores"]) else ""
            dados.append(registro)
        tabelas.append(pd.DataFrame(dados))

# Concatenar e remover duplicatas
if tabelas:
    df_final = pd.concat(tabelas, ignore_index=True)
    antes = len(df_final)
    df_final = df_final.drop_duplicates()
    depois = len(df_final)
    removidas = antes - depois

    print(f"\n=== TABELA FINAL ===")
    print(f"Linhas totais: {antes} | Duplicatas removidas: {removidas} | Linhas únicas: {depois}")
    display(df_final.reset_index(drop=True))
else:
    print("Nenhuma tabela gerada.")

--- AgedPayables_20260207170508.parquet | AgedPayables | Período: 2026-02-07 ---
  Linhas extraídas: 5
--- AgedPayables_20260207170624.parquet | AgedPayables | Período: 2026-02-07 ---
  Linhas extraídas: 5
--- AgedPayables_20260325043000.parquet | AgedPayables | Período: 2026-03-01 ---
  Linhas extraídas: 2
--- AgedPayables_20260325043100.parquet | AgedPayables | Período: 2026-03-01 ---
  Linhas extraídas: 2
--- AgedPayables_20260325214230.parquet | AgedPayables | Período: 2026-03-25 ---
  Linhas extraídas: 5

=== TABELA FINAL ===
Linhas totais: 19 | Duplicatas removidas: 8 | Linhas únicas: 11


,Grupo,Vendor,Current,1 - 30,31 - 60,61 - 90,91 and over,Total,Account
0,None,Brosnahan Insurance Agency,,,,241.23,,241.23,NaN
1,None,Diego's Road Warrior Bodyshop,,,755.00,,,755.00,NaN
2,None,Norton Lumber and Building Materials,,,,205.00,,205.00,NaN
3,None,PG&E,,,,,86.44,86.44,NaN
4,None,Robertson & Associates,,,,315.00,,315.00,NaN
5,Category 1,NaN,NaN,NaN,NaN,NaN,NaN,1050.00,Line 1
6,Category 1,NaN,NaN,NaN,NaN,NaN,NaN,515.00,Line 1B
7,None,Brosnahan Insurance Agency,,,,,241.23,241.23,NaN
8,None,Diego's Road Warrior Bodyshop,,,,755.00,,755.00,NaN
9,None,Norton Lumber and Building Materials,,,,,205.00,205.00,NaN


In [24]:
# Ver colunas e primeiras linhas da tabela final
print(f"Colunas: {list(df_final.columns)}\n")
print(df_final.head(3).to_string())

Colunas: ['Grupo', 'Vendor', 'Current', '1 - 30', '31 - 60', '61 - 90', '91 and over', 'Total', 'Account']

  Grupo                                Vendor Current 1 - 30 31 - 60 61 - 90 91 and over   Total Account
0  None            Brosnahan Insurance Agency                         241.23              241.23     NaN
1  None         Diego's Road Warrior Bodyshop                 755.00                      755.00     NaN
2  None  Norton Lumber and Building Materials                         205.00              205.00     NaN


In [25]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
